In [ ]:

# 1. IMPORTS

import os, cv2, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from torchvision.models import resnet50
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 2. PATHS

kvasir_img_dir = r"E:\Projects\Project 5 - YoLo Clonoscopy Paper\Code\Dataset\Kvasir-SEG\images"
kvasir_mask_dir = r"E:\Projects\Project 5 - YoLo Clonoscopy Paper\Code\Dataset\Kvasir-SEG\masks"

cvc_img_dir = r"E:\Projects\Project 5 - YoLo Clonoscopy Paper\Code\Dataset\CVC\PNG\Original"


# 3. DATASETS

class PolypDataset(Dataset):
    def __init__(self, img_dir, mask_dir, files):
        self.files = files
        self.img_dir = img_dir
        self.mask_dir = mask_dir

        self.transform = A.Compose([
            A.Resize(512,512),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.Normalize(mean=(0.485,0.456,0.406),
                        std=(0.229,0.224,0.225)),
            ToTensorV2()
        ])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]

        img = cv2.imread(os.path.join(self.img_dir, f))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(os.path.join(self.mask_dir, f), 0)
        mask = (mask>0).astype(np.float32)

        aug = self.transform(image=img, mask=mask)
        return aug['image'], aug['mask'].unsqueeze(0)

class CVC_Dataset(Dataset):
    def __init__(self, img_dir):
        self.files = sorted(os.listdir(img_dir))
        self.img_dir = img_dir

        self.transform = A.Compose([
            A.Resize(512,512),
            A.Normalize(mean=(0.485,0.456,0.406),
                        std=(0.229,0.224,0.225)),
            ToTensorV2()
        ])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]
        img = cv2.imread(os.path.join(self.img_dir, f))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        aug = self.transform(image=img)
        return aug['image'], f


# 4. SPLIT

files = sorted(os.listdir(kvasir_img_dir))
train_f, test_f = train_test_split(files, test_size=0.1, random_state=42)
train_f, val_f = train_test_split(train_f, test_size=0.2, random_state=42)

train_loader = DataLoader(PolypDataset(kvasir_img_dir,kvasir_mask_dir,train_f), batch_size=8, shuffle=True)
val_loader   = DataLoader(PolypDataset(kvasir_img_dir,kvasir_mask_dir,val_f), batch_size=8)

cvc_loader = DataLoader(CVC_Dataset(cvc_img_dir), batch_size=8) if os.path.exists(cvc_img_dir) else None


# 5. BASELINE MODELS

class UNET(nn.Module):
    def __init__(self):
        super().__init__()
        b = resnet50(weights="IMAGENET1K_V1")
        self.encoder = nn.Sequential(b.conv1,b.bn1,b.relu,b.maxpool,b.layer1,b.layer2,b.layer3)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(1024,512,2,2), nn.ReLU(),
            nn.ConvTranspose2d(512,256,2,2), nn.ReLU(),
            nn.ConvTranspose2d(256,64,2,2), nn.ReLU(),
            nn.Conv2d(64,1,1)
        )

    def forward(self,x):
        x = self.encoder(x)
        return torch.sigmoid(self.decoder(x))


# 6. PROPOSED MODEL

class DualPathModel(nn.Module):
    def __init__(self):
        super().__init__()
        b = resnet50(weights="IMAGENET1K_V1")

        self.shared = nn.Sequential(
            b.conv1,b.bn1,b.relu,b.maxpool,b.layer1,b.layer2
        )

        self.det_head = nn.Conv2d(512,5,1)
        self.encoder = b.layer3

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(1024,512,2,2), nn.ReLU(),
            nn.ConvTranspose2d(512,256,2,2), nn.ReLU(),
            nn.ConvTranspose2d(256,64,2,2), nn.ReLU(),
            nn.Conv2d(64,1,1)
        )

    def forward(self,x):
        shared = self.shared(x)
        det = self.det_head(shared)
        seg = torch.sigmoid(self.decoder(self.encoder(shared)))
        return det, seg


# 7. LOSSES

bce = nn.BCELoss()

def dice(pred,gt):
    smooth=1e-5
    inter=(pred*gt).sum()
    return 1-(2*inter+smooth)/(pred.sum()+gt.sum()+smooth)

def seg_loss(p,g):
    p = F.interpolate(p,size=g.shape[2:])
    return bce(p,g)+dice(p,g)

def total_loss(det,seg,gt):
    return 0.6*det.mean()+0.4*seg_loss(seg,gt)


# 8. METRICS

def compute_metrics(pred, gt):
    pred = (pred>0.5).float()

    TP = (pred*gt).sum()
    FP = (pred*(1-gt)).sum()
    FN = ((1-pred)*gt).sum()

    precision = TP/(TP+FP+1e-6)
    recall    = TP/(TP+FN+1e-6)
    dice_score= (2*TP)/(2*TP+FP+FN+1e-6)
    iou       = TP/(TP+FP+FN+1e-6)

    return precision.item(), recall.item(), dice_score.item(), iou.item()


# 9. TRAIN FUNCTION

def train_model(model, epochs=20):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)

    for e in range(epochs):
        model.train()
        loss_sum=0

        for img,mask in train_loader:
            img,mask = img.to(device),mask.to(device)

            if isinstance(model, DualPathModel):
                det,seg = model(img)
                loss = total_loss(det,seg,mask)
            else:
                seg = model(img)
                loss = seg_loss(seg,mask)

            opt.zero_grad()
            loss.backward()
            opt.step()

            loss_sum += loss.item()

        print(f"{model.__class__.__name__} Epoch {e}: {loss_sum/len(train_loader):.4f}")


# 10. EVALUATION

def evaluate_model(model):
    model.eval()

    p_list,r_list,d_list,i_list = [],[],[],[]

    with torch.no_grad():
        for img,mask in val_loader:
            img,mask = img.to(device),mask.to(device)

            if isinstance(model, DualPathModel):
                _,seg = model(img)
            else:
                seg = model(img)

            seg = F.interpolate(seg,size=mask.shape[2:])
            p,r,d,i = compute_metrics(seg,mask)

            p_list.append(p); r_list.append(r)
            d_list.append(d); i_list.append(i)

    print(f"\n{model.__class__.__name__} Results:")
    print("Precision:", np.mean(p_list))
    print("Recall:", np.mean(r_list))
    print("Dice:", np.mean(d_list))
    print("IoU:", np.mean(i_list))
    
# 11. GRAD-CAM

class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.layer = layer
        self.grad=None; self.act=None

        layer.register_forward_hook(lambda m,i,o: setattr(self,'act',o))
        layer.register_backward_hook(lambda m,gi,go: setattr(self,'grad',go[0]))

    def generate(self, img):
        img = img.unsqueeze(0).to(device)
        img.requires_grad=True

        _,seg = self.model(img)
        loss = seg.mean()

        self.model.zero_grad()
        loss.backward()

        w = torch.mean(self.grad, dim=(2,3), keepdim=True)
        cam = torch.sum(w*self.act, dim=1)

        cam = F.relu(cam)
        cam = F.interpolate(cam.unsqueeze(1),(512,512)).squeeze().cpu().numpy()
        cam = (cam-cam.min())/(cam.max()-cam.min()+1e-8)

        return cam

def visualize_gradcam(model):
    dataset = PolypDataset(kvasir_img_dir,kvasir_mask_dir,val_f)
    cam = GradCAM(model, model.shared[-1])

    for i in range(3):
        img,_ = dataset[i]
        heat = cam.generate(img)

        img_np = img.permute(1,2,0).numpy()
        img_np = (img_np-img_np.min())/(img_np.max()-img_np.min())

        overlay = 0.6*img_np + 0.4*cv2.applyColorMap((heat*255).astype(np.uint8),cv2.COLORMAP_JET)/255

        plt.figure(figsize=(9,3))
        plt.subplot(1,3,1); plt.imshow(img_np); plt.title("Original"); plt.axis('off')
        plt.subplot(1,3,2); plt.imshow(heat,cmap='jet'); plt.title("GradCAM"); plt.axis('off')
        plt.subplot(1,3,3); plt.imshow(overlay); plt.title("Overlay"); plt.axis('off')
        plt.show()


# 12. RUN

unet = UNET().to(device)
proposed = DualPathModel().to(device)

train_model(unet)
train_model(proposed)

evaluate_model(unet)
evaluate_model(proposed)

visualize_gradcam(proposed)

if cvc_loader:
    print("\nCVC dataset inference done.")